# Lab Day 10: Finetune Qwen3-VL-4B (QLoRA) để định vị hư hại mặt đường
Notebook triển khai các bước trong GUIDE.md. Giữ nguyên các file đề bài và chỉ dùng số liệu sinh từ lần chạy thật.

**Colab setup:** clone or upload the complete repository into the runtime and run this notebook from the repository root; the notebook imports `lab_day10_solution.py` and calls the unchanged `scripts/` tools.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU (16 GB)
from pathlib import Path
required = [Path("scripts/prepare_data.py"), Path("scripts/score.py"), Path("lab_day10_solution.py")]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise RuntimeError("Colab cần toàn bộ repository tại thư mục hiện tại; thiếu: " + ", ".join(missing))
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

# Drive is used only for adapter/projector backups.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
kaggle_username = userdata.get('KAGGLE_USERNAME')
kaggle_key = userdata.get('KAGGLE_KEY')
if not kaggle_username or not kaggle_key:
    raise RuntimeError('Hãy thêm KAGGLE_USERNAME và KAGGLE_KEY trong Colab Secrets.')
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_key

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
from lab_day10_solution import verify_dataset, visualize_ground_truth



checked = verify_dataset()  # version, exact split/class counts, and session isolation

visualize_ground_truth(checked['splits']['train'], count=6)

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from lab_day10_solution import yolo_to_box, objects_to_answer

from scripts.labcommon import read_jsonl



example = read_jsonl('data/splits/train.jsonl')[0]

print('Grounding target:', objects_to_answer(example['objects']))

print('One converted box:', yolo_to_box(example['objects'][0]['label'], example['objects'][0]['yolo']) if example['objects'] else '[]')

## 3. E0: zero-shot

In [ ]:
from lab_day10_solution import run_zero_shot



# Generates work/e0.jsonl from all 311 official test images.

run_zero_shot()

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --split test --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from lab_day10_solution import inspect_masking_and_collation



# Verifies assistant-only labels, JSON + <|im_end|>, image tensor concatenation,

# and reports the visual token count produced by this installed processor.

processor = inspect_masking_and_collation()

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from lab_day10_solution import apply_setup



# The concrete setup is applied by the training loop, then the injected module

# count is checked against 252 LLM targets (E1/E2) or 348 total (E3).

SETUP_DESCRIPTIONS = {

    'E1': 'LLM LoRA; ViT and Projector frozen',

    'E2': 'E1 + full float32 visual.merger and deepstack mergers',

    'E3': 'E2 + ViT LoRA on qkv / linear_fc1 / linear_fc2',

}

for name, description in SETUP_DESCRIPTIONS.items():

    print(name, '—', description)

## 6. Vòng huấn luyện

In [ ]:
from lab_day10_solution import train_supervised_sweep



# Trains E1, E2, E3 for 300 optimizer steps each; batch=2, accumulation=4.

# Selects the best setup using validation F1 only and backs up every 100 steps.

sft_runs = train_supervised_sweep()

## 7. Suy luận và chấm E1–E3

In [ ]:
from lab_day10_solution import evaluate_supervised



# Reloads each adapter (including separately saved projector weights), evaluates

# the official 311-image test split, writes e1/e2/e3 evidence, and draws 6 examples.

test_scores = evaluate_supervised()

for name, score in test_scores.items():

    print(name, score['micro']['f1'], score['f1_ci95'], score['parse_valid_rate'])

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
from lab_day10_solution import check_iou_reward_cases, run_rank_ablation, run_grpo



# E4 changes only LoRA rank from the best validation setup; alpha remains 2*rank.

e4_scores = run_rank_ablation()



# E5 is optional. The five deterministic reward cases can be checked without GPU training.

reward_cases = check_iou_reward_cases()

RUN_E5 = False  # Set True only to attempt the optional 60-step TRL GRPO run on T4.

if RUN_E5:

    e5_score = run_grpo(steps=60)



# Before submission, complete REPORT.md with your actual evidence and run:

# python scripts/check_submission.py
